# Part 2: Multi-Source RAG with Routing

This notebook implements a RAG system that intelligently routes queries to different data sources:
- **CSV (Structured):** Daily sales data — 1000 rows, 35 products, 5 regions, Oct-Dec 2024
- **Text (Unstructured):** 10 product pages with descriptions, specs, and reviews

## Architecture
```
User Query --> Query Router --> Data Source(s) --> LLM Answer
               (classify)      CSV | Text | Both   (with Context)

  Analytical/numerical  --> CSV (pandas)
  Product details/reviews --> Unstructured text (semantic search)
  Both                  --> Combine both sources
```

## Setup

In [11]:
import os
import logging
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
import litellm

# LangChain for semantic search
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load environment variables
load_dotenv(Path('../.env'))

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configuration
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
MODEL = 'groq/llama-3.1-8b-instant'

DATA_DIR = Path('../data').resolve()
CSV_PATH = DATA_DIR / 'structured' / 'daily_sales.csv'
TEXT_DIR = DATA_DIR / 'unstructured'

print(f"Groq API Key loaded: {'Yes' if GROQ_API_KEY else 'No'}")
print(f"CSV path: {CSV_PATH} (exists: {CSV_PATH.exists()})")
print(f"Text dir: {TEXT_DIR} (exists: {TEXT_DIR.exists()})")
print(f"Text files: {len(list(TEXT_DIR.glob('*.txt')))}")

Groq API Key loaded: Yes
CSV path: /Users/akshayarun/Desktop/Spring_2026/DSAN-6725/spring-2026-a03-akshay17arun/data/structured/daily_sales.csv (exists: True)
Text dir: /Users/akshayarun/Desktop/Spring_2026/DSAN-6725/spring-2026-a03-akshay17arun/data/unstructured (exists: True)
Text files: 10


## Step 1: Load Data Sources

In [12]:
# Load CSV data
df = pd.read_csv(CSV_PATH)
df['date'] = pd.to_datetime(df['date'])
print(f"CSV shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Categories: {df['category'].unique().tolist()}")
print(f"Regions: {df['region'].unique().tolist()}")
print(f"\nFirst 3 rows:")
df.head(3)

CSV shape: (1000, 8)
Date range: 2024-10-03 00:00:00 to 2024-12-31 00:00:00
Categories: ['Books', 'Electronics', 'Food & Grocery', 'Home & Kitchen', 'Pet Supplies', 'Office Supplies', 'Toys & Games', 'Sports & Outdoors', 'Beauty & Personal Care', 'Clothing']
Regions: ['Central', 'West', 'North', 'South', 'East']

First 3 rows:


,date,product_id,product_name,category,units_sold,unit_price,total_revenue,region
0,2024-10-03,BOOK003,Mystery Novel Collection,Books,42,24.99,1049.58,Central
1,2024-10-03,ELEC002,USB-C Fast Charger,Electronics,57,24.99,1424.43,Central
2,2024-10-03,BOOK001,Python Programming Guide,Books,39,39.65,1546.35,West


In [13]:
# Load unstructured text files
text_docs = {}
for txt_file in sorted(TEXT_DIR.glob('*.txt')):
    text_docs[txt_file.stem] = txt_file.read_text()
    print(f"Loaded {txt_file.name} ({len(text_docs[txt_file.stem])} chars)")

print(f"\nTotal text documents: {len(text_docs)}")

Loaded BEAU001_product_page.txt (2833 chars)
Loaded BOOK001_product_page.txt (2740 chars)
Loaded CLTH001_product_page.txt (2543 chars)
Loaded ELEC001_product_page.txt (2325 chars)
Loaded FOOD001_product_page.txt (2732 chars)
Loaded HOME003_product_page.txt (2437 chars)
Loaded OFFC001_product_page.txt (2674 chars)
Loaded PETS001_product_page.txt (2694 chars)
Loaded SPRT001_product_page.txt (2387 chars)
Loaded TOYS001_product_page.txt (2566 chars)

Total text documents: 10


## Step 2: Build Vector Store for Semantic Search

We chunk and embed the product text files to enable semantic search over product descriptions and reviews.

In [14]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Chunk the documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

all_chunks = []
for doc_name, text in text_docs.items():
    # Extract product name and SKU from file name for metadata
    chunks = splitter.create_documents(
        [text],
        metadatas=[{'source': doc_name, 'file': f'{doc_name}.txt'}]
    )
    all_chunks.extend(chunks)

print(f"Total chunks: {len(all_chunks)}")
print(f"Example chunk (first 300 chars):\n{all_chunks[0].page_content[:300]}")

Total chunks: 47
Example chunk (first 300 chars):
VITAMIN C SERUM - PRODUCT PAGE

Product: Vitamin C Serum
Brand: GlowLab Skincare
Price: $28.99
SKU: BEAU001
Category: Beauty & Personal Care

PRODUCT DESCRIPTION:
Reveal brighter, more youthful skin with GlowLab's Vita


In [15]:
# Build FAISS vector store with HuggingFace embeddings
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'}
)

print("Building vector store...")
vectorstore = FAISS.from_documents(all_chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={'k': 5})

print(f"Vector store built with {len(all_chunks)} chunks.")

2026-03-05 20:16:00,121 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13380.00it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-05 20:16:00,787 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Building vector store...
Vector store built with 47 chunks.


## Step 3: Query Router

The router classifies each query to determine which source(s) to use.

In [16]:
def classify_query(question: str) -> str:
    """
    Route the query to the appropriate data source(s).

    Returns one of:
    - 'csv': Analytical/numerical questions about sales data
    - 'text': Questions about product features, descriptions, reviews
    - 'both': Questions requiring insights from both sources
    """
    q = question.lower()

    # Analytical indicators (CSV)
    csv_keywords = [
        'revenue', 'sales', 'total', 'average', 'highest', 'lowest', 'most',
        'units sold', 'volume', 'region', 'month', 'december', 'october',
        'november', 'quarter', 'trend', 'how many', 'how much', 'price'
    ]

    # Product detail indicators (Text)
    text_keywords = [
        'feature', 'review', 'customer', 'specification', 'spec', 'description',
        'quality', 'rating', 'pros', 'cons', 'ease of', 'cleaning', 'comfort',
        'recommend', 'warranty', 'material', 'color', 'size', 'weight'
    ]

    # Multi-source indicators (Both)
    both_keywords = [
        'best', 'sell well', 'highly rated', 'popular', 'suggest', 'which product'
    ]

    csv_score = sum(1 for kw in csv_keywords if kw in q)
    text_score = sum(1 for kw in text_keywords if kw in q)
    both_score = sum(1 for kw in both_keywords if kw in q)

    # Explicit both-source queries
    if both_score > 0 or (csv_score > 0 and text_score > 0):
        return 'both'

    if csv_score > text_score:
        return 'csv'
    elif text_score > 0:
        return 'text'
    else:
        # Default to text for ambiguous queries about products
        return 'text'


# Test routing on all 6 questions
test_questions = [
    "What was the total revenue for Electronics category in December 2024?",
    "Which region had the highest sales volume?",
    "What are the key features of the Wireless Bluetooth Headphones?",
    "What do customers say about the Air Fryer's ease of cleaning?",
    "Which product has the best customer reviews and how well is it selling?",
    "I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?"
]

print("Query routing results:")
for i, q in enumerate(test_questions, 1):
    route = classify_query(q)
    print(f"  Q{i} [{route:5s}]: {q[:65]}")

Query routing results:
  Q1 [csv  ]: What was the total revenue for Electronics category in December 2
  Q2 [csv  ]: Which region had the highest sales volume?
  Q3 [text ]: What are the key features of the Wireless Bluetooth Headphones?
  Q4 [text ]: What do customers say about the Air Fryer's ease of cleaning?
  Q5 [both ]: Which product has the best customer reviews and how well is it se
  Q6 [both ]: I want a product for fitness that is highly rated and sells well 


## Step 4: Source-Specific Retrieval

Each source type has its own retrieval strategy.

In [17]:
def retrieve_from_csv(question: str) -> str:
    """
    Retrieve relevant data from the CSV using pandas.
    Applies filters and aggregations based on the question.
    """
    q = question.lower()
    context_parts = []

    # Always include schema info
    context_parts.append(f"CSV Schema: {', '.join(df.columns.tolist())}")
    context_parts.append(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
    context_parts.append(f"Regions: {', '.join(df['region'].unique())}")
    context_parts.append(f"Categories: {', '.join(df['category'].unique())}")

    # Category filter
    category_map = {
        'electronics': 'Electronics', 'books': 'Books', 'sports': 'Sports',
        'beauty': 'Beauty', 'clothing': 'Clothing', 'home': 'Home & Kitchen',
        'toys': 'Toys & Games', 'office': 'Office', 'pets': 'Pet Supplies',
        'food': 'Food & Gourmet'
    }
    selected_category = None
    for kw, cat in category_map.items():
        if kw in q:
            selected_category = cat
            break

    # Date filter
    month_map = {'october': 10, 'november': 11, 'december': 12}
    selected_month = None
    for month_name, month_num in month_map.items():
        if month_name in q:
            selected_month = month_num
            break

    # Region filter
    region_map = {'north': 'North', 'south': 'South', 'east': 'East', 'west': 'West', 'central': 'Central'}
    selected_region = None
    for kw, reg in region_map.items():
        if kw in q:
            selected_region = reg
            break

    # Apply filters
    filtered = df.copy()
    if selected_category:
        filtered = filtered[filtered['category'] == selected_category]
        context_parts.append(f"\nFiltered to category: {selected_category}")
    if selected_month:
        filtered = filtered[filtered['date'].dt.month == selected_month]
        context_parts.append(f"Filtered to month: {selected_month} (2024)")
    if selected_region:
        filtered = filtered[filtered['region'] == selected_region]
        context_parts.append(f"Filtered to region: {selected_region}")

    # Compute relevant aggregations
    if 'revenue' in q or 'total' in q:
        if selected_category or selected_month:
            total = filtered['total_revenue'].sum()
            context_parts.append(f"\nTotal revenue (filtered): ${total:,.2f}")
        # By category
        by_cat = df.groupby('category')['total_revenue'].sum().sort_values(ascending=False)
        context_parts.append(f"\nTotal revenue by category:\n{by_cat.to_string()}")
        if selected_month:
            month_df = df[df['date'].dt.month == selected_month]
            by_cat_month = month_df.groupby('category')['total_revenue'].sum().sort_values(ascending=False)
            context_parts.append(f"\nRevenue by category (month {selected_month}):\n{by_cat_month.to_string()}")

    if 'volume' in q or 'units' in q or 'sales' in q or 'highest' in q or 'most' in q:
        # By region
        by_region = df.groupby('region')['units_sold'].sum().sort_values(ascending=False)
        context_parts.append(f"\nTotal units sold by region:\n{by_region.to_string()}")
        # By product
        by_product = df.groupby('product_name')['units_sold'].sum().sort_values(ascending=False).head(10)
        context_parts.append(f"\nTop 10 products by units sold:\n{by_product.to_string()}")
        # By category
        by_cat_vol = df.groupby('category')['units_sold'].sum().sort_values(ascending=False)
        context_parts.append(f"\nUnits sold by category:\n{by_cat_vol.to_string()}")

    if selected_region and ('sell' in q or 'sales' in q or 'revenue' in q or 'recommend' in q):
        region_data = df[df['region'] == selected_region]
        by_prod = region_data.groupby('product_name').agg(
            units_sold=('units_sold', 'sum'),
            total_revenue=('total_revenue', 'sum'),
            category=('category', 'first')
        ).sort_values('units_sold', ascending=False).head(15)
        context_parts.append(f"\nTop products in {selected_region} region:\n{by_prod.to_string()}")

    # Sample rows
    sample = filtered.head(10).to_string(index=False)
    context_parts.append(f"\nSample rows (up to 10):\n{sample}")

    return '\n'.join(context_parts)


def retrieve_from_text(question: str) -> str:
    """
    Retrieve relevant product information using semantic search.
    """
    docs = retriever.invoke(question)
    context_parts = []
    for doc in docs:
        source = doc.metadata.get('source', 'unknown')
        context_parts.append(f"--- Source: {source} ---\n{doc.page_content}")
    return '\n\n'.join(context_parts)


def retrieve_context(question: str, route: str) -> dict:
    """
    Route to the right data source(s) and return combined context.
    """
    if route == 'csv':
        csv_ctx = retrieve_from_csv(question)
        return {
            'context': f"[SOURCE: CSV Sales Data]\n{csv_ctx}",
            'sources_used': ['csv']
        }
    elif route == 'text':
        text_ctx = retrieve_from_text(question)
        return {
            'context': f"[SOURCE: Product Text Files]\n{text_ctx}",
            'sources_used': ['text']
        }
    else:  # 'both'
        csv_ctx = retrieve_from_csv(question)
        text_ctx = retrieve_from_text(question)
        combined = (
            f"[SOURCE 1: CSV Sales Data]\n{csv_ctx}\n\n"
            f"[SOURCE 2: Product Text Files]\n{text_ctx}"
        )
        return {
            'context': combined,
            'sources_used': ['csv', 'text']
        }


print("Retrieval functions defined.")

Retrieval functions defined.


## Step 5: Answer Generation

In [18]:
def generate_answer(question: str, context: str, sources_used: list) -> str:
    """Generate an answer using the LLM with the retrieved context."""

    source_desc = ' + '.join(sources_used)
    system_prompt = f"""You are a helpful e-commerce analytics assistant. You have access to two data sources:
1. CSV Sales Data: 1000 daily sales records (Oct-Dec 2024), 35 products, 5 regions
2. Product Text Files: Detailed product descriptions, specifications, and customer reviews

For this question, you are using: {source_desc}

Your answer should:
- Directly answer the question with specific numbers, product names, or details
- Reference the data source (CSV or product page) for key facts
- Be clear and well-structured
- For recommendations, explain your reasoning based on both sales data and reviews"""

    user_prompt = f"""Question: {question}

Retrieved context:
---
{context[:12000]}
---

Please answer the question based on the above context."""

    response = litellm.completion(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=1500
    )
    return response.choices[0].message.content


def answer_question(question: str) -> dict:
    """Full pipeline: route -> retrieve -> generate."""
    print(f"\n{'='*70}")
    print(f"Question: {question}")
    print('='*70)

    route = classify_query(question)
    print(f"Route: {route}")

    print("Retrieving context...")
    retrieval = retrieve_context(question, route)
    context = retrieval['context']
    sources_used = retrieval['sources_used']
    print(f"Sources: {sources_used} | Context: {len(context)} chars")

    print("Generating answer...")
    answer = generate_answer(question, context, sources_used)

    print(f"\nAnswer:\n{answer}")

    return {
        'question': question,
        'route': route,
        'sources_used': sources_used,
        'context_length': len(context),
        'answer': answer
    }


print("Answer generation functions defined.")

Answer generation functions defined.


## Step 6: Answer All Test Questions

In [19]:
import time

test_questions = [
    "What was the total revenue for Electronics category in December 2024?",
    "Which region had the highest sales volume?",
    "What are the key features of the Wireless Bluetooth Headphones?",
    "What do customers say about the Air Fryer's ease of cleaning?",
    "Which product has the best customer reviews and how well is it selling?",
    "I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?"
]

results = []
for i, question in enumerate(test_questions, 1):
    print(f"\n\n{'#'*70}")
    print(f"# Question {i}/{len(test_questions)}")
    print('#'*70)
    result = answer_question(question)
    results.append(result)
    time.sleep(10)

print("\n\nAll 6 questions answered!")

20:16:01 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
2026-03-05 20:16:01,513 - INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq




######################################################################
# Question 1/6
######################################################################

Question: What was the total revenue for Electronics category in December 2024?
Route: csv
Retrieving context...
Sources: ['csv'] | Context: 2513 chars
Generating answer...


20:16:01 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
2026-03-05 20:16:01,807 - INFO - Wrapper: Completed Call, calling success_handler



Answer:
The total revenue for the Electronics category in December 2024 was $142,864.31.


20:16:11 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
2026-03-05 20:16:11,824 - INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq


...


Answer:
Based on the provided context, the product with the best customer reviews is the "Mystery Novel Collection" (BOOK001) with an average rating of 4.6/5 from 2,234 reviews.

As for its sales performance, according to the CSV Sales Data, BOOK001 has sold a total of 42 units on October 3, 2024, in the Central region, and 39 units on the same day in the West region. Additionally, it sold 1 unit on October 3, 2024, in the South region.

Here's a breakdown of BOOK001's sales data:

- Total units sold: 42 (Central) + 39 (West) + 1 (South) = 82 units
- Total revenue: Not provided in the sample data, but can be calculated by multiplying the units sold by the unit price.

To determine the overall sales performance of BOOK001, we would need to analyze the sales data for the entire date range (October 3 to December 31, 2024). However, based on the provided sample data, it appears that BOOK001 is a popular product with strong customer reviews and decent sales performance.


20:16:54 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq
2026-03-05 20:16:54,323 - INFO - 
LiteLLM completion() model= llama-3.1-8b-instant; provider = groq




######################################################################
# Question 6/6
######################################################################

Question: I want a product for fitness that is highly rated and sells well in the West region. What do you recommend?
Route: both
Retrieving context...
Sources: ['csv', 'text'] | Context: 6356 chars
Generating answer...


20:16:55 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
2026-03-05 20:16:55,267 - INFO - Wrapper: Completed Call, calling success_handler



Answer:
Based on the sales data and product reviews, I recommend the "Running Shoes Men" (product_id: CLTH001) for fitness in the West region.

**Sales Data:**
The "Running Shoes Men" product has sold 301 units in the West region, generating a total revenue of $27,086.99. This is the second-best-selling product in the West region, after the "Denim Jeans Classic" (product_id: CLTH004).

**Product Reviews:**
The reviews for the "Running Shoes Men" product are overwhelmingly positive, with an average rating of 4.67 stars out of 5. Reviewers praise the shoes for their comfort, traction, and value for money. One reviewer, Chris L., even used the shoes for a half marathon and reported no blisters or hot spots.

**Key Features:**
The "Running Shoes Men" product features a responsive foam midsole for energy return, a breathable engineered mesh upper, and a reinforced heel counter for stability. The shoes also have a rubber outsole with a multi-directional traction pattern and reflective detai

## Step 7: Save Results to part2_results.txt

In [20]:
output_path = Path('../part2_results.txt')

with open(output_path, 'w') as f:
    f.write("# Part 2: Multi-Source RAG with Routing\n")
    f.write("# E-Commerce Analytics: CSV Sales Data + Product Text Files\n")
    f.write("=" * 70 + "\n\n")

    for i, result in enumerate(results, 1):
        f.write(f"{'='*70}\n")
        f.write(f"Question {i}: {result['question']}\n")
        f.write(f"{'='*70}\n\n")
        f.write(f"Route: {result['route']}\n")
        f.write(f"Sources Used: {', '.join(result['sources_used'])}\n")
        f.write(f"Context Retrieved: {result['context_length']} characters\n\n")
        f.write(f"Answer:\n{result['answer']}\n\n")

print(f"Results saved to {output_path.resolve()}")
print(f"File size: {output_path.stat().st_size:,} bytes")

Results saved to /Users/akshayarun/Desktop/Spring_2026/DSAN-6725/spring-2026-a03-akshay17arun/part2_results.txt
File size: 7,170 bytes
